Setting up a training loop of Neural KEM with patches

this is an implementation of the following manuscript:

S. Li, K. Gong, R. D. Badawi, E. J. Kim, J. Qi and G. Wang, "Neural KEM: A Kernel Method With Deep Coefficient Prior for PET Image Reconstruction," in IEEE Transactions on Medical Imaging, vol. 42, no. 3, pp. 785-796, March 2023, doi: 10.1109/TMI.2022.3217543

This demo is a jupyter notebook, i.e. intended to be run step by step.

Author: Daniel Deidda

First version: 13th of Nov 2023

CCP SyneRBI Synergistic Image Reconstruction Framework (SIRF).
Copyright 2023 NPL.
SPDX-License-Identifier: Apache-2.0

# Setting up the training


In [ ]:
# Get the parent directory
import sys
import os
import pathlib

#parent_dir = os.path.dirname(os.path.realpath('.'))
notebooks_dir = pathlib.Path().parent.absolute() 
parent_dir =  os.path.dirname(notebooks_dir)
print(parent_dir, notebooks_dir)
# Add the parent directory to sys.path
sys.path.append(parent_dir)

In [ ]:
# Import the PET reconstruction engine
import sirf.STIR as pet
# Set the verbosity
pet.set_verbosity(0)
# Store temporary sinograms in RAM
pet.AcquisitionData.set_storage_scheme("memory")
# SIRF STIR message redirector
import sirf

import sirf.STIR as pet
msg = sirf.STIR.MessageRedirector(info=None, warn=None, errr=None)
# Load dataset and model
from torchKernel.kernel.LHK import  BuildK
from torchKernel.architectures.UNet import  UNet
from torchKernel.utils.torch_operations import  tdivide
from torchKernel.utils.plots import plot_many_tensors, plot_many_numpys
from torchKernel.utils.sirf_torch import primal_op as F
from torchKernel.utils.sirf_torch import dual_op as B
from torchKernel.utils.system import create_working_dir_and_move_into
from torchKernel.utils.analytics import estimate_MSE_and_save, plot_losses

import os
import numpy as np
import time
import torch

from torchKernel.algorithms.Algorithm import Algorithm

from tqdm.auto import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# the following data is obtained by running:
# python algorithms/Brain_simulation.py --counts_ratio=1

working_dir=create_working_dir_and_move_into(parent_dir)
sinogram_template = pet.AcquisitionData('FDG_tumour_sino_2d_noisy_seed0.hs');

sinogram_template_l = pet.AcquisitionData('FDG_tumour_sino_2d_noisy_ratio_0_1_seed0.hs');

y=torch.from_numpy(sinogram_template.as_array()).repeat(1,1,1,1).to(device)
y_low=torch.from_numpy(sinogram_template_l.as_array()).repeat(1,1,1,1).to(device)

true_brain = pet.ImageData('FDG_tumour_2d.hv') #(144x144)
umap = pet.ImageData('uMap_2d.hv')
T1 = pet.ImageData('T1_2d.hv')
image_template = true_brain#.zoom_image(zooms=(2,2,2))

true_brain_np = true_brain.as_array()
true_brain_tt = torch.from_numpy(true_brain_np).repeat(1,1,1,1).to(device)# tt torch tensor
xt=true_brain_tt

T1_np = T1.as_array()
anat_tt0 = torch.from_numpy(T1_np).repeat(1,1,1,1).to(device)
anat_tt = torch.from_numpy(T1_np/T1_np.max()).repeat(1,1,1,1).to(device)
anat=T1/T1.max()

umap_np = umap.as_array()
umap_tt = torch.from_numpy(umap_np).repeat(1,1,1,1).to(device)

algo=Algorithm(1, 1, anat, sinogram_template, sinogram_template.get_uniform_copy(1), sinogram_template.get_uniform_copy(0),
                         umap, format, 1,0, 0, 200,
                         0, 0)
# acq_model = get_acquisition_model(sinogram_template, umap)
plot_many_tensors(0 , true_brain_tt.max(), 0,'jet', [true_brain_tt,anat_tt0],['GT','MR'])


In [ ]:
#set up model
# create initial image estimate

f = algo.fp
b = algo.bp

prior=anat_tt
input_im=prior.to(device)
ref = xt.to(device)#torch.reshape(ref,(1,4,4,4))
xr=torch.rand(ref.shape)


In [ ]:

print(device)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) # if you are using multi-GPU.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
#set parameters
w=9
k=48
ksigma=4
isVox=False
hybrid=False
LR = 1e-3
#create kernel

# BK=BuildK(self.ksigma,self.is_voxelised,self.save_mem_k,[umap.spacing[0], umap.spacing[1], umap.spacing[2]])

BK=BuildK(ksigma,0,0,[umap.spacing[0], umap.spacing[1], umap.spacing[2]])
Kw , ID = BK(anat_tt,k,w)

#run nKEM
net = UNet(1,16,1).to(device)

data_log = { }
data_log["loss"] = [ ]
data_log['epoch'] = [ ]

num_out_iter=60
num_deep_iter=150 #150
num_em_iter=4
p = pk = []
p += [x for x in net.parameters() ]
optimiser = torch.optim.Adam(p, lr=LR)
Poisson_loss = torch.nn.PoissonNLLLoss(log_input=False,reduction='none').to(device)
alpha=torch.ones(xt.shape, device=device)
net_out=torch.ones(xt.shape, device=device)
net_in = prior.to(device)

for i in tqdm(range(num_out_iter)):
    with torch.no_grad():
        for it in range(num_em_iter):
            
            ys = y
            sens = b(torch.ones(ys.shape, device=device))
            ksens=BK.kernelise_image_t(Kw,ID,sens)
            ka= BK.kernelise_image(Kw,ID,alpha)
            fka=f(ka).to(device)
            grad=b((tdivide(ys,fka)))
            kgrad=BK.kernelise_image_t(Kw,ID,grad)
            curr_kem_i =  tdivide(alpha,ksens)*kgrad
            alpha =curr_kem_i

        alpha.requires_grad_(True)

   
    for j in range(num_deep_iter): 

        optimiser.zero_grad() #clear old grads 
        # Forward  
        net_out = net(net_in)

        # compute loss
        tot_loss = torch.mean(sens * Poisson_loss(net_out, curr_kem_i))

        # backprop
        tot_loss.backward()

        # `clip grad: always after backward() and before step()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1) 

        # update weights
        optimiser.step() 
            
        data_log["loss"].append(tot_loss.item())
        data_log['epoch'].append(len(data_log["loss"]))  
    
    #save things
    l=BK.kernelise_image(Kw,ID,net_out)
    # print ('outer iter '+str(i))
    string = 'nKEM_k'+str(k)+'_w'+str(w)+'_dit'+str(num_deep_iter)+'_eit'+str(num_em_iter)+'_tit'  
    torch.save({'model_state_dict_net': net.state_dict(),
                'optimizer_state_dict': optimiser.state_dict(),
                }, 'trained_extra.torch_model')
            
    np.save(string+str(i)+'.npy',alpha.detach().cpu().numpy())
    np.save(string+'_data_log',data_log)
        
    

plot_many_tensors(0 , true_brain_tt.max(), 0,'jet', [l,alpha],['image','alpha'])
  

In [ ]:
#loss
loss_name1 = string
plot_losses([loss_name1])